# College Stuff+ (whiff-based)
Author: Zach Campbell

Stuff+ built from pitch characteristics only (no location). The model predicts the probability of a whiff on a swing,
which is the outcome most driven by pitch shape and velocity. Predictions are out-of-fold by pitcher, so no pitcher is
scored by a model that trained on him. 100 = average college pitch, 10 = one standard deviation; higher is better.

In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

mov = pd.read_csv("MovementP4.csv", low_memory=False)
rv = pd.read_csv("run_values.csv")

## Label each pitch
Event names must match `run_values.csv` exactly, or the merge drops the pitch.

In [6]:
def map_event(result):
    r = str(result).lower()
    if "intentional" in r or "interference" in r or r in ("unknown", "nan"): return None
    if r.startswith("strikeout"): return "strikeout"
    if r == "strike looking": return "called_strike"
    if r == "strike swinging": return "swinging_strike"
    if r.startswith("foul"): return "foul"
    if r.startswith("home run"): return "home_run"
    if r.startswith("triple"): return "triple"
    if r.startswith("double play"): return "field_out"  # must come before "double"
    if r.startswith("double"): return "double"
    if r.startswith("single") or r.startswith("reached on error"): return "single"
    if r == "walk": return "walk"
    if r == "hit by pitch": return "hit_by_pitch"
    if r.startswith("ball"): return "ball"
    if "out" in r or "sac" in r or "fielder" in r or r.startswith("in play"): return "field_out"
    return None

counts = mov["count"].str.split("-", expand=True)
mov["balls"] = pd.to_numeric(counts[0], errors="coerce")
mov["strikes"] = pd.to_numeric(counts[1], errors="coerce")
mov = mov[mov.balls.between(0, 3) & mov.strikes.between(0, 2)]
mov["event"] = mov.pitchResult.map(map_event)

shape = ["releaseVelocity", "Spin", "IndVertBrk", "HorzBrk", "Extension", "RelX", "RelZ", "SpinDir"]
mov[shape] = mov[shape].apply(pd.to_numeric, errors="coerce")

# Drop untracked pitches rather than filling them with averages
df = mov.merge(rv, on=["event", "balls", "strikes"]).dropna(subset=shape).reset_index(drop=True)

result = df.pitchResult.str.lower()
df["whiff"] = result.isin(["strike swinging", "strikeout (swinging)"]).astype(int)
in_play = df.event.isin(["field_out", "single", "double", "triple", "home_run"])
df["swing"] = (df.whiff.eq(1) | df.event.eq("foul") | in_play).astype(int)
df["bunt"] = result.str.contains("bunt")
df = df.rename(columns={"pitcherAbbrevName": "name", "fullName": "college"})
print(df.event.value_counts())

event
ball               178592
called_strike       82220
foul                81489
field_out           53861
swinging_strike     37240
strikeout           30134
single              21040
walk                13645
double               5415
hit_by_pitch         4191
home_run             3682
triple                498
Name: count, dtype: int64


## Features
Lefties are mirrored so a slider looks the same from either side. Each pitch is also compared to the pitcher's own fastball.

In [7]:
lefty = df.pitcherHand.eq("L")
df["RelX_n"] = np.where(lefty, -df.RelX, df.RelX)
df["HorzBrk_n"] = np.where(lefty, -df.HorzBrk, df.HorzBrk)
df["SpinDir_n"] = np.where(lefty, 360 - df.SpinDir, df.SpinDir)
df["same_side"] = (df.pitcherHand == df.batterHand).astype(int)

fb_cols = ["releaseVelocity", "IndVertBrk", "HorzBrk_n", "RelZ", "RelX_n"]
fb = df[df.pitchType.isin(["FA", "FF", "SI"])].groupby("pitcherId")[fb_cols].mean().add_prefix("fb_")
df = df.join(fb, on="pitcherId")
for name, col in [("velo", "releaseVelocity"), ("ivb", "IndVertBrk"), ("hb", "HorzBrk_n"), ("relz", "RelZ"), ("relx", "RelX_n")]:
    df[f"{name}_diff"] = df[col] - df[f"fb_{col}"]
df["ptype"] = df.pitchType.astype("category")

features = ["releaseVelocity", "Spin", "IndVertBrk", "HorzBrk_n", "Extension", "RelX_n", "RelZ", "SpinDir_n",
            "same_side", "ptype", "velo_diff", "ivb_diff", "hb_diff", "relz_diff", "relx_diff"]

## Whiff model
Five folds grouped by pitcher: each pitcher's pitches are scored by a model that never saw him.

In [8]:
params = dict(n_estimators=500, learning_rate=0.03, num_leaves=31, min_child_samples=500,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5, verbose=-1)
swings = (df.swing.eq(1) & ~df.bunt).to_numpy()
df["xwhiff"] = 0.0
for train, test in GroupKFold(5).split(df, groups=df.pitcherId):
    train = train[swings[train]]
    model = lgb.LGBMClassifier(**params).fit(df.iloc[train][features], df.iloc[train].whiff, categorical_feature=["ptype"])
    df.loc[test, "xwhiff"] = model.predict_proba(df.iloc[test][features])[:, 1]

print(f"Whiff-on-swing AUC: {roc_auc_score(df.whiff[swings], df.xwhiff[swings]):.3f}")

Whiff-on-swing AUC: 0.645


## Does it predict the future?
Split each pitcher's games into odd and even. A good Stuff+ from one half should predict whiffs and run value in the other,
and should beat the pitcher's own run value from that half (which is mostly noise).

In [9]:
df["half"] = df.gameId.rank(method="dense") % 2
agg = lambda d: d.groupby(["pitcherId", "pitchType"]).agg(
    xwhiff=("xwhiff", "mean"), rv=("delta_run_exp", "mean"), whiffs=("whiff", "sum"), swings=("swing", "sum"), n=("whiff", "size"))
halves = agg(df[df.half == 0]).join(agg(df[df.half == 1]), lsuffix="_a", rsuffix="_b", how="inner")
halves = halves[(halves.n_a >= 50) & (halves.n_b >= 50)]

print(f"{len(halves)} pitcher/pitch-type groups with 50+ pitches in each half")
print(f"Stuff (half A) vs whiff%  (half B): {halves.xwhiff_a.corr(halves.whiffs_b / halves.swings_b):+.3f}")
print(f"Stuff (half A) vs run value (half B): {-halves.xwhiff_a.corr(halves.rv_b):+.3f}")
print(f"Raw run value A vs B (baseline):      {halves.rv_a.corr(halves.rv_b):+.3f}")

1273 pitcher/pitch-type groups with 50+ pitches in each half
Stuff (half A) vs whiff%  (half B): +0.692
Stuff (half A) vs run value (half B): +0.239
Raw run value A vs B (baseline):      +0.118


## Stuff+ and output
Breaking balls get more whiffs than fastballs by nature, so Stuff+ is scaled within each pitch type:
a 110 fastball is one standard deviation better than the average college fastball.
A pitcher's overall Stuff+ is the usage-weighted average of his pitches.

In [10]:
df["ptype_group"] = df.pitchType.replace({"FF": "FA"})  # FF and FA are both four-seamers
df = df[~df.ptype_group.isin(["UN", "IN", "PO"])]       # unknown, intentional ball, pitchout
by_type = df.groupby("ptype_group").xwhiff
df["StuffPlus"] = 100 + 10 * (df.xwhiff - by_type.transform("mean")) / by_type.transform("std")

df[["uniqPitchId", "pitcherId", "name", "college", "team", "pitchType", "StuffPlus", "xwhiff"] + shape].to_csv(
    "StuffPlus_Predictions.csv", index=False)

by_pitch = (df.groupby(["pitcherId", "name", "college", "team", "ptype_group"])
              .agg(pitches=("StuffPlus", "size"), StuffPlus=("StuffPlus", "mean"), velo=("releaseVelocity", "mean"),
                   ivb=("IndVertBrk", "mean"), hb=("HorzBrk_n", "mean"))
              .query("pitches >= 100").round(1).sort_values("StuffPlus", ascending=False).reset_index())
by_pitch.to_csv("StuffPlus_ByPitch.csv", index=False)

# One row per pitcher: overall Stuff+, plus Stuff+ and usage for each pitch he throws
keys = ["pitcherId", "name", "college", "team", "pitcherHand"]
by_pitcher = (df.groupby(keys).agg(pitches=("StuffPlus", "size"), StuffPlus=("StuffPlus", "mean"))
                .query("pitches >= 300"))
per_type = df.groupby(keys + ["ptype_group"]).StuffPlus.agg(["mean", "size"]).unstack("ptype_group")
usage = per_type["size"].div(per_type["size"].sum(axis=1), axis=0)
by_pitcher = by_pitcher.join(per_type["mean"].where(usage >= 0.05).add_suffix("_Stuff+"))
by_pitcher = by_pitcher.join(usage.where(usage >= 0.05).mul(100).add_suffix("_usage%"))
by_pitcher = by_pitcher.dropna(axis=1, how="all").round(1).sort_values("StuffPlus", ascending=False).reset_index()
by_pitcher.to_csv("StuffPlus_ByPitcher.csv", index=False)
by_pitcher.head(20)

,pitcherId,name,college,team,pitcherHand,pitches,StuffPlus,CH_Stuff+,CU_Stuff+,FA_Stuff+,...,FS_Stuff+,SI_Stuff+,SL_Stuff+,CH_usage%,CU_usage%,FA_usage%,FC_usage%,FS_usage%,SI_usage%,SL_usage%
0,1190560000,C. Meyer,West Virginia University,WVU,R,758,119.1,103.0,131.0,105.5,...,NaN,NaN,130.2,6.3,32.1,35.4,NaN,NaN,NaN,21.9
1,1307169536,J. Mayers,Louisiana State University,LSU,R,309,116.8,NaN,NaN,118.0,...,95.0,NaN,118.6,NaN,NaN,78.6,NaN,5.2,NaN,13.3
2,1694435683,L. Kelly,Arizona State University,ASU,R,461,113.5,NaN,NaN,114.8,...,NaN,113.8,109.4,NaN,NaN,65.5,NaN,NaN,16.3,14.5
3,1134108077,N. Williams,Mississippi State University,MSST,R,461,113.3,NaN,117.3,111.6,...,NaN,NaN,NaN,NaN,30.8,67.9,NaN,NaN,NaN,NaN
4,1190987520,C. Shores,Louisiana State University,LSU,R,1104,112.6,108.7,NaN,106.4,...,NaN,116.4,113.2,6.3,NaN,23.4,NaN,NaN,39.2,31.1
5,1357477972,C. Townsend,University of Mississippi,MISS,R,514,111.7,NaN,118.0,109.6,...,NaN,NaN,111.4,NaN,11.1,46.3,31.3,NaN,NaN,7.2
6,1124908800,C. Wheatley,University of Iowa,IOWA,R,396,111.7,NaN,NaN,106.5,...,NaN,114.5,110.6,NaN,NaN,8.8,NaN,NaN,44.7,38.9
7,1229119488,C. Spencer,University of Mississippi,MISS,R,392,111.7,NaN,NaN,113.3,...,NaN,NaN,105.3,NaN,NaN,80.4,NaN,NaN,NaN,13.8
8,1309413120,L. Dotson,Mississippi State University,MSST,L,363,111.4,103.9,NaN,113.7,...,NaN,NaN,93.5,6.6,NaN,84.8,NaN,NaN,NaN,7.7
9,1136535572,L. Broderick,University of Nebraska,NEB,R,650,111.1,NaN,113.6,102.0,...,NaN,NaN,115.6,NaN,6.0,31.2,NaN,NaN,NaN,59.4
